# Live method registration

This notebook demonstrates how analysis code can be registered and replaced while a notebook is running.

Features demonstrated:

- replacing an existing `.m` method with a notebook-local implementation;
- building an analysis method from a numerical fitting function;
- creating and using a custom DataFrame accessor;
- registering a notebook-local plotting overlay.

## Import

In [ ]:
import numpy as np
import pandas as pd
from matplotlib.patches import Rectangle

from labdata import methods, plots
from labdata.tools import fit2

%load_ext labdata.notebook

In [ ]:
with np.load("data/demo3.npz", allow_pickle=False) as d:
    DATA = pd.DataFrame({
        "FILE": d["FILE"],
        "IMAGE": list(d["IMAGE"]),
        "METADATA//CAMERA": d["METADATA//CAMERA"],
        "METADATA//ATOM": d["METADATA//ATOM"],
        "METADATA//ROI": [tuple(roi) for roi in d["METADATA//ROI"]],
        "lattice_depth[Er]": d["lattice_depth[Er]"],
    })

## Data

In [ ]:
DATA.p.table(x="lattice_depth[Er]", width=120, ncols=11)
DATA.p.imshow(vmin=0, vmax=1.2, cmap="turbo")

## Overwriting a method

The replacement calls the numerical array fit from `labdata.tools.fit2`, expands its result into record fields, and adds new metrics based on local information (camera, atom). Registering the same name replaces `DATA.m.gauss1dfit` for the current Python process and prints a notice; the package source is unchanged. Besides adding local special behaviour, this is also useful for debugging and testing changes to methods that are already part of the package.

In [ ]:
PIXEL_SIZE = 6.5e-6
MAGNIFICATION = 0.75
WAVELENGTH = 766.700921822e-9

PIXEL_SCALE = PIXEL_SIZE / MAGNIFICATION
RESONANT_CROSS_SECTION = 3 * WAVELENGTH**2 / (2 * np.pi)

In [ ]:
@methods.register
@methods.method(vectorise=True)
def gauss1dfit(data, **kwargs):
    image = data["IMAGE"]
    if "MASK" in data:
        image = np.where(data["MASK"], image, np.nan)
    result = fit2.gauss1dfit(image, **kwargs)
    roi = data.get("METADATA", {}).get("ROI", (0, image.shape[0], 0, image.shape[1]))

    for name in ("var", "uvar"):
        values = result[name].copy()
        values["x_centre"] += roi[2]
        values["y_centre"] += roi[0]
        values = {
            f"{key}[pixel]" if any(quantity in key for quantity in ("centre", "sigma")) else key: value
            for key, value in values.items()
        }
        values.update({
            "x_centre[m]": values["x_centre[pixel]"] * PIXEL_SCALE,
            "y_centre[m]": values["y_centre[pixel]"] * PIXEL_SCALE,
            "x_sigma[m]": values["x_sigma[pixel]"] * PIXEL_SCALE,
            "y_sigma[m]": values["y_sigma[pixel]"] * PIXEL_SCALE,
            "sigma[m]": values["sigma[pixel]"] * PIXEL_SCALE,
            "atomnum": (values["pixelsum"] * PIXEL_SCALE**2 / RESONANT_CROSS_SECTION
            ),
        })
        data.setdefault(name, {}).update(values)
    data.update({key: value for key, value in result.items() if key not in ("var", "uvar")})
    return data

In [ ]:
d = DATA.m.gauss1dfit()

In [ ]:
d.p.table(x="lattice_depth[Er]", width=80, ncols=11)
d.p.implot("XS", fit="x_fit")

## Custom accessor

Supplying `accessor="lattice"` creates a separate namespace and registers `visibility` on it. The method sums two equal-sized regions, stores their definitions under `INFO`, and adds their values and the derived scalar field to each record. Repeated values of the same lattice depth can be passed directly to `errorbar`, which plots their mean and standard deviation.

In [ ]:
VISIBILITY_ROIS = {
    "peak": (40, 56, 63, 79),
    "diagonal": (56, 72, 56, 72),
}


@methods.register(accessor="lattice")
@methods.method(vectorise=True)
def visibility(data, rois):
    values = {}
    for name, (row_start, row_stop, column_start, column_stop) in rois.items():
        image = data["IMAGE"][row_start:row_stop, column_start:column_stop]
        values[name] = np.sum(image)

    var = {
        "peak_pixelsum": values["peak"],
        "diagonal_pixelsum": values["diagonal"],
        "visibility": (values["peak"] - values["diagonal"]) / (values["peak"] + values["diagonal"]),
    }
    info = {**data.get("INFO", {}), "visibility_rois": dict(rois)}
    return {**data, "var": {**data.get("var", {}), **var}, "INFO": info}

In [ ]:
d = DATA.lattice.visibility(VISIBILITY_ROIS)

In [ ]:
%print d

In [ ]:
d.p.plot(x="lattice_depth[Er]", y="var//visibility")

In [ ]:
d.p.errorbar(x="lattice_depth[Er]", y="var//visibility", capsize=3)

## Custom plot

Plot functions can also be registered on the same custom accessor. `subplot` pairs each record with the corresponding axis created by `table`; this `.lattice` overlay reads the ROI definitions retained by `visibility`.

In [ ]:
@plots.register(accessor="lattice")
@plots.subplot
def visibility_rois(data, *, ax):
    colors = {"peak": "white", "diagonal": "magenta"}
    for name, color in colors.items():
        row_start, row_stop, column_start, column_stop = data[f"INFO//visibility_rois//{name}"]
        ax.add_patch(Rectangle(
            (column_start, row_start), column_stop - column_start, row_stop - row_start,
            fill=False, edgecolor=color, linewidth=1.5,
        ))

In [ ]:
d.p.table(x="lattice_depth[Er]", width=80, ncols=11)
d.p.imshow(vmin=0, vmax=1.2, cmap="turbo")
d.lattice.visibility_rois()